In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
from sklearn.model_selection import train_test_split


In [3]:
df_all = pd.read_csv("train_with_clusters.csv")
cluster0 = df_all[df_all["cluster"] == 0].copy()

print(cluster0["Bankrupt?"].value_counts())   # sanity check 0/1 counts

Bankrupt?
0    596
1    132
Name: count, dtype: int64


In [4]:
# Features for this subgroup:
# The 40-feature list from joblib.

top40 = joblib.load("top_features_for_clustering.joblib")  # list of column names

X_sub = cluster0[top40].copy()
y_sub = cluster0["Bankrupt?"].copy()

print(X_sub.shape, y_sub.value_counts())

(728, 40) Bankrupt?
0    596
1    132
Name: count, dtype: int64


In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X_sub, y_sub,
    test_size=0.2,
    random_state=42,
    stratify=y_sub
)

scaler = StandardScaler().fit(X_train)

X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)

In [7]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

rf  = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

gb  = GradientBoostingClassifier(
    random_state=42
)

et  = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

base_estimators = [
    ("rf", rf),
    ("gb", gb),
    ("et", et)
]

meta = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

stack = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)


In [8]:
from sklearn.metrics import recall_score, make_scorer

recall1 = make_scorer(recall_score, pos_label=1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    stack,
    X_sub.values,  # you can scale inside a Pipeline later; for now use scaler+model manually
    y_sub.values,
    cv=cv,
    scoring=recall1,
    n_jobs=-1
)

print("Mean Eq(1) (recall for y=1) across CV:", scores.mean())
print("CV scores:", scores)


Mean Eq(1) (recall for y=1) across CV: 0.7116809116809116
CV scores: [0.57692308 0.81481481 0.66666667 0.61538462 0.88461538]


In [9]:
# For tree models you can skip scaling; if you keep it, apply to whole X_sub.
X_full_s = scaler.transform(X_sub)

stack.fit(X_full_s, y_sub)


StackingClassifier(estimators=[('rf',
                                RandomForestClassifier(class_weight='balanced_subsample',
                                                       n_estimators=300,
                                                       n_jobs=-1,
                                                       random_state=42)),
                               ('gb',
                                GradientBoostingClassifier(random_state=42)),
                               ('et',
                                ExtraTreesClassifier(class_weight='balanced_subsample',
                                                     n_estimators=300,
                                                     n_jobs=-1,
                                                     random_state=42))],
                   final_estimator=LogisticRegression(class_weight='balanced',
                                                      max_iter=1000,
                                                      random_state=42),
                   n_jobs=-1, stack_method='predict_proba')

In [10]:
from sklearn.metrics import confusion_matrix

y_pred = stack.predict(X_full_s)

cm = confusion_matrix(y_sub, y_pred, labels=[0, 1])
FF, FT, TF, TT = cm.ravel()

print("Confusion matrix [ [FF, FT], [TF, TT] ]:\n", cm)
print("FF:", FF, "FT:", FT, "TF:", TF, "TT:", TT)

eq1 = TT / (TF + TT) if (TF + TT) > 0 else 0.0
print("Eq(1) accuracy for cluster 0:", eq1)


Confusion matrix [ [FF, FT], [TF, TT] ]:
 [[596   0]
 [  0 132]]
FF: 596 FT: 0 TF: 0 TT: 132
Eq(1) accuracy for cluster 0: 1.0


In [11]:
import joblib

cluster0_model = {
    "cluster_id": 0,
    "feature_names": top40,
    "scaler": scaler,
    "model": stack
}

joblib.dump(cluster0_model, "cluster0_stacking_A.joblib")


['cluster0_stacking_A.joblib']